# Wiki Scrape

a script to scrape star names off of wikipedia's list of stars, therefore assuring that data on pageviews can be found, and clean the data

In [87]:
## imports

import requests
import random
import pandas as pd
from io import StringIO

# Data Extraction

In [88]:
## constants

HEADERS = {'User-Agent': 'CelestialCelebs (sw2499@cornell.edu)'}

In [34]:
## a function to get data from wikipedia api
#
# params:
#   PAGE: the name of the Wikipedia page to fetch data for
#   FILE_PATH: the path to the file where the fetched data will be saved
# returns:
#   None

def get_wiki_data(page, file_path):
    r = requests.get(
        'https://en.wikipedia.org/w/api.php',
        params={'action': 'parse', 'format': 'json', 'page': page, 'prop': 'text', 'formatversion': 2},
        headers=HEADERS,
        timeout=60
    )
    r.raise_for_status()
    data = r.json()['parse']['text']

    # write to a file so we can inspect data without repeatedly scraping
    f = open(file_path, 'w')
    f.write(data)
    f.close()

In [ ]:
## get wiki data for stars, exoplanets, and galaxies
# uncomment if stars.html, exoplanets.html, or galaxies.html are not present in the repo

# get_wiki_data('List_of_proper_names_of_stars', 'stars.html')
# get_wiki_data('List_of_proper_names_of_exoplanets', 'exoplanets.html')
# get_wiki_data('List_of_galaxies', 'galaxies.html')
# get_wiki_data('List_of_exceptional_asteroids', 'asteroids.html')

# Data Parsing

## Stars

main useful info: star name (if there is one) and official designation

In [89]:
## read data

star_data = open('stars.html', 'r').read()

## random seed

RANDOM_SEED = 8
random.seed(RANDOM_SEED)

In [60]:
## parse html data and fix star names and designations

stars_df = pd.read_html(StringIO(star_data), match='Modern proper name', extract_links='body')[0] # read first table matching "proper name"
N_STARS = stars_df.shape[0] # total number of rows
stars_df['Link'] = '' # by default assume there's no link

for r in range(1, N_STARS):
    row = stars_df.loc[r]
    
    if ((row['Modern proper name'][1] is not None) and
        (row['Modern proper name'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        stars_df.loc[r, 'Link'] = row['Modern proper name'][1]
    stars_df.loc[r, 'Modern proper name'] = row['Modern proper name'][0] # fix name
    
    if ((row['Designation'][1] is not None) and
        (row['Designation'][1][:6] == '/wiki/')):
        # if the designation is linked, the entry is linked
        stars_df.loc[r, 'Link'] = row['Designation'][1]
    stars_df.loc[r, 'Designation'] = row['Designation'][0] # fix designation

In [ ]:
## a function to randomly select stars from the table, given that the star has a valid hyperlink to another page
#
# params:
#   n: number of random stars to select
# returns:
#   2D list containing [common_name, designation, wikipedia link] for each selected star

def select_random_stars(n):
    objs = []
    row_idxs = random.sample(range(1, N_STARS), n)
    for idx in row_idxs:
        # get row from table
        row = stars_df.loc[idx]
        if (row['Link'] != ''):
            objs.append([row['Modern proper name'], row['Designation'], row['Link']])
        
    return pd.DataFrame(objs, columns=['common_name', 'designation', 'link'])

In [ ]:
## select 200 stars and save as a csv
# uncomment line below if stars.csv doesn't exist in the repo
# select_random_stars(200).to_csv('stars.csv', index=False)
# file exists now, don't need to run it again

## Exoplanets

main useful info: planet name and host star name

WIP WIP WIP WIP

In [101]:
## read data

exoplanet_data = open('exoplanets.html', 'r').read()

In [102]:
## parse html data and fix exoplanet names and designations

planets_df = pd.read_html(StringIO(exoplanet_data), match='Name of planet', extract_links='body')[0] # read first table matching "name of planet"
N_PLANETS = planets_df.shape[0] # total number of rows
planets_df['Linked'] = False # by default assume there's no link

for r in range(1, N_PLANETS):
    row = planets_df.loc[r]
    
    if ((row['Name of planet'][1] is not None) and
        (row['Name of planet'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        planets_df.loc[r, 'Linked'] = True
    planets_df.loc[r, 'Name of planet'] = row['Name of planet'][0] # fix name

    # if star is linked, that doesn't mean the planet is linked, but we can fix the formatting
    planets_df.loc[r, 'Name of star'] = row['Name of star'][0] # fix name of star

In [103]:
## a function to randomly select exoplanets from the table, given that the exoplanet has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [common name, host star] for each selected exoplanet

def select_random_planets(n):
    objs = []
    row_idxs = random.sample(range(1, N_PLANETS), n)
    for idx in row_idxs:
        # get row from table
        row = planets_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Name of planet'], row['Name of star']])
        
    return pd.DataFrame(objs, columns=['common_name', 'host_star'])

In [104]:
## select 75 exoplanets and save as a csv
select_random_planets(75).to_csv('exoplanets.csv', index=False)

## Galaxies

main useful info: name

In [71]:
## read data

galaxy_data = open('galaxies.html', 'r').read()

In [72]:
## parse html data and fix exoplanet names and designations

galaxies_df = pd.read_html(StringIO(galaxy_data), match='Galaxy', extract_links='body')[0] # read first table matching "galaxy"
N_GALAXIES = galaxies_df.shape[0] # total number of rows
galaxies_df['Linked'] = False # by default assume there's no link

for r in range(1, N_GALAXIES):
    row = galaxies_df.loc[r]
    
    if ((row['Galaxy'][1] is not None) and
        (row['Galaxy'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        galaxies_df.loc[r, 'Linked'] = True
    galaxies_df.loc[r, 'Galaxy'] = row['Galaxy'][0] # fix name

In [73]:
## a function to randomly select galaxies from the table, given that the galaxy has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [official name] for each selected exoplanet

def select_random_galaxies(n):
    objs = []
    row_idxs = random.sample(range(1, N_GALAXIES), n)
    for idx in row_idxs:
        # get row from table
        row = galaxies_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Galaxy']])
        
    return pd.DataFrame(objs, columns=['official_name'])

In [ ]:
## select 75 galaxies and save as a csv
select_random_galaxies(50).to_csv('galaxies.csv', index=False)

## Asteroids

main useful info: name

In [36]:
## read data

asteroids = open('asteroids.html', 'r').read()

In [47]:
## parse html data and fix asteroid names

asteroids_df = pd.read_html(StringIO(asteroids), match='Name', extract_links='body')[0] # read first table matching "name"
N_ASTEROIDS = asteroids_df.shape[0] # total number of rows
asteroids_df['Linked'] = False # by default assume there's no link

for r in range(0, N_ASTEROIDS):
    row = asteroids_df.loc[r]
    
    if ((row['Name'][1] is not None) and
        (row['Name'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        asteroids_df.loc[r, 'Linked'] = True
    asteroids_df.loc[r, 'Name'] = row['Name'][0] # fix name

# remove irrelevant rows - see original wikipedia page for details
asteroids_df = asteroids_df[:40]
N_ASTEROIDS = asteroids_df.shape[0] # total number of rows

In [53]:
## a function to return all asteroids that have a valid hyperlink to another page
#
# params:
#   n: number of random asteroids to select
# returns:
#   list containing [official name] for each selected asteroid

def select_asteroids(n):
    objs = []
    for idx in range(0, N_ASTEROIDS):
        # get row from table
        row = asteroids_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Name']])
        
    return pd.DataFrame(objs, columns=['official_name'])

In [54]:
## select the 40 asteroids and save as a csv
select_asteroids(40).to_csv('asteroids.csv', index=False)

# Data Cleaning

## Stars

astroquery has trouble with non-English characters

In [95]:
stars_csv = open('stars.csv', 'r').read()
cleaned_stars = []

# replacements for greek characters in star designations
greek_replacements = {
    'α': 'alf',
    'β': 'bet',
    'η': 'eta',
    'θ': 'tet',
    'ο': 'omi',
    'τ': 'tau',
    'υ': 'ups',
    'γ': 'gam',
    'κ': 'kap',
    'λ': 'lam',
    'μ': 'mu',
    'ν': 'nu',
    'ξ': 'xi',
    'ρ': 'rho',
    'σ': 'sig',
    'φ': 'phi',
    'χ': 'chi',
    'ψ': 'psi',
    'ω': 'ome',
    'ε': 'eps',
    'δ': 'del',
    'ζ': 'zet',
    'ι': 'iota',
    'π': 'pi',
    'ς': 'sig'}

# replacements for non-english characters in common names and star designations
char_replacements = {
    'á': 'a',
    'é': 'e',
    'í': 'i',
    'ó': 'o',
    'ú': 'u',
    'ä': 'a',
    'ö': 'o',
    'ï': 'i',
    'ü': 'u',
    'ë': 'e',
    'č': 'c',
    'ỹ': 'y',
    'â': 'a',
    'ã': 'a',
    'ũ̀': 'u',
    '†': ''
}

for char in char_replacements.keys():
    stars_csv = stars_csv.replace(char, char_replacements[char])

stars_csv = stars_csv.splitlines()
for i in range(1, len(stars_csv)):
    line = stars_csv[i]
    [common_name, designation, link] = line.strip().split(',')
    
    # extract wiki page name
    link = link[6:]

    # replace greek characters
    if (ord(designation[0]) >= 128): # english character
        designation = designation.replace(designation[0], greek_replacements[designation[0]])

    # handle multiple-star systems
    words = designation.split(' ')
    if ((words[-1] == 'A') or (words[-1] == 'Aa') or (words[-1] == 'Aa1')):
        designation = ' '.join(words[:-1])

    cleaned_stars.append([common_name, designation, link])

In [96]:
pd.DataFrame(cleaned_stars, columns=['common_name', 'designation', 'page_name']).to_csv('clean_stars.csv', index=False)
# all data processing will be done with clean_stars.csv

## Exoplanets

In [29]:
exoplanets_csv = open('exoplanets.csv', 'r').read()
cleaned_exoplanets = []

# replacements
replacements = {
    'á': 'a',
    'é': 'e',
    'í': 'i',
    'ó': 'o',
    'ú': 'u',
    'ä': 'a',
    'ö': 'o',
    'ï': 'i',
    'ü': 'u',
    'ë': 'e',
    'č': 'c',
    'ỹ': 'y',
    'â': 'a',
    'ã': 'a'
}

# replace bad characters
for char in replacements.keys():
    exoplanets_csv = exoplanets_csv.replace(char, replacements[char])

# clean data and separate star name from designation
lines = exoplanets_csv.splitlines()
for line in lines:
    if (line == 'common_name,host_star'):
        continue
    else:
        [common_name, host_star] = line.strip().split(',')
        [designation, star_name] = host_star.split(' (')
        star_name = star_name[:-1] # remove trailing ')'
        cleaned_exoplanets.append([common_name, designation, star_name])

cleaned_exoplanets = pd.DataFrame(cleaned_exoplanets, columns=['common_name', 'star_designation', 'host_star'])

In [31]:
cleaned_exoplanets.to_csv('clean_exoplanets.csv', index=False)

galaxies and asteroids do not need data cleaning - data is already clean